# C.3 robustness probe v4 — confirm the wikitext_base graduation signal

**Why this notebook.** The v3 follow-up at n=10 produced CI-disjoint signal at `wikitext_base` (lr_pull=0.1, n_events=1000) in two strata (calibrated/tight Δ +0.083, default/spread Δ +0.055). Before claiming Phase 3 graduation, three orthogonal robustness probes:

| tag | varies | n_seeds | purpose |
|---|---|---:|---|
| `replicate_seeds10_19` | new seed set (10..19) at exact same config | 10 | seed-specificity check |
| `beta30_seeds0_9` | β = 30 (canonical retrieval β per `STATUS.md` operational policy) instead of 10 | 10 | retrieval-temperature robustness |
| `D2048_seeds0_9` | D = 2048 instead of 4096 | 10 | substrate-dim robustness |

All else fixed at the v3 graduation cell: `wikitext, lr_pull=0.1, n_events=1000, alpha_anti=0.01, repulsion_step_size=0.05, window=8, vocab_cap=1000`.

**Robustness criterion (binding for the writeup):**
- 3 / 3 probes CI-disjoint in any stratum  →  Phase 3 graduation locked.
- 2 / 3 probes CI-disjoint                →  graduation with one caveat, narrow the operating-point claim in the report.
- 1 / 3 probes CI-disjoint                →  result is operating-point-specific; reframe v3 as a positive *operating-point sensitivity finding* rather than graduation.
- 0 / 3 probes CI-disjoint                →  v3 result is artifact; do NOT graduate.

**3 conditions × 10 seeds = 30 parallel CUDA subprocesses.** Reuses the v3 reliability machinery (3 patches embedded, smoke test, per-seed fan-out, auto-log-dump on failure, aggregation cell with Wilson CIs).

In [ ]:
# 1. Clone the repo and apply three patches:
#    (a) --lr-pull / --lr-push CLI flags in the C.3 driver.
#    (b) Kernel-trick eigvalsh fix in consolidation.py (CUDA-stable).
#    (c) Switch wikitext loader to "Salesforce/wikitext" namespace.
%cd /content
!rm -rf Neuro-AI
!git clone https://github.com/Dypatterson/Neuro-AI.git
%cd Neuro-AI
!git checkout codex/phase5-prime-bundle-first-scene-memory
!git log --oneline -3

import subprocess
pre_pull = subprocess.run(['grep', '-c', '--', '--lr-pull', 'experiments/c3_phase3_exit_criterion.py'], capture_output=True, text=True).stdout.strip()
pre_gram = subprocess.run(['grep', '-c', 'kernel trick', 'src/energy_memory/phase4/consolidation.py'], capture_output=True, text=True).stdout.strip()
pre_wt   = subprocess.run(['grep', '-c', 'Salesforce/wikitext', 'src/energy_memory/phase2/corpus.py'], capture_output=True, text=True).stdout.strip()
print(f'pre-patch tracers: --lr-pull = {pre_pull}; kernel-trick gram = {pre_gram}; Salesforce/wikitext = {pre_wt}')

patch = r'''diff --git a/experiments/c3_phase3_exit_criterion.py b/experiments/c3_phase3_exit_criterion.py
--- a/experiments/c3_phase3_exit_criterion.py
+++ b/experiments/c3_phase3_exit_criterion.py
@@ -576,6 +576,8 @@ def _run_single_seed_condition(
     k: int,
     alpha_anti: float,
     repulsion_step_size: float,
+    lr_pull: float,
+    lr_push: float,
     device: str,
     repo_root: Path,
     wikitext_corpus: Optional[_WikiTextCorpus] = None,
@@ -756,6 +758,8 @@ def _run_single_seed_condition(
             vocab_size=vocab_size,
             n_events=n_consolidation_events,
             device=device,
+            lr_pull=lr_pull,
+            lr_push=lr_push,
             repulsion_step_size=repulsion_step_size,
         )
 
@@ -838,6 +842,8 @@ def run(
     n_consolidation_events: int = 1000,
     alpha_anti: float = 0.0,
     repulsion_step_size: float = 0.0,
+    lr_pull: float = 0.1,
+    lr_push: float = 0.05,
     device: str,
     output_dir: Path,
     repo_root: Path,
@@ -919,6 +925,8 @@ def run(
                     k=k,
                     alpha_anti=alpha_anti,
                     repulsion_step_size=repulsion_step_size,
+                    lr_pull=lr_pull,
+                    lr_push=lr_push,
                     device=device,
                     repo_root=repo_root,
                     wikitext_corpus=wikitext_corpus,
@@ -1010,6 +1018,8 @@ def run(
             "substrate_repulsion_active": bool(
                 alpha_anti > 0.0 and repulsion_step_size > 0.0
             ),
+            "lr_pull": float(lr_pull),
+            "lr_push": float(lr_push),
             "operating_point": {
                 "D": D,
                 "landscape_size": landscape_size,
@@ -1370,6 +1380,27 @@ def main(argv: Optional[Sequence[str]] = None) -> int:
             "smoke (no inter-atom-separability force)."
         ),
     )
+    parser.add_argument(
+        "--lr-pull",
+        type=float,
+        default=0.1,
+        help=(
+            "Per-event consolidation pull learning rate (OnlineCodebookUpdater "
+            "lr_pull). Default 0.1 matches the existing Path α smoke. Sweep "
+            "above this to test whether consolidation strength is too weak "
+            "to express corpus-specific learning at the synthetic operating "
+            "point."
+        ),
+    )
+    parser.add_argument(
+        "--lr-push",
+        type=float,
+        default=0.05,
+        help=(
+            "Per-event consolidation push learning rate (OnlineCodebookUpdater "
+            "lr_push). Default 0.05 matches the existing Path α smoke."
+        ),
+    )
     parser.add_argument(
         "--repulsion-step-size",
         type=float,
@@ -1468,6 +1499,8 @@ def main(argv: Optional[Sequence[str]] = None) -> int:
         n_consolidation_events=args.n_consolidation_events,
         alpha_anti=args.alpha_anti,
         repulsion_step_size=args.repulsion_step_size,
+        lr_pull=args.lr_pull,
+        lr_push=args.lr_push,
         device=args.device,
         output_dir=output_dir,
         repo_root=repo_root,
diff --git a/src/energy_memory/phase4/consolidation.py b/src/energy_memory/phase4/consolidation.py
--- a/src/energy_memory/phase4/consolidation.py
+++ b/src/energy_memory/phase4/consolidation.py
@@ -639,10 +639,29 @@ class ConsolidationState:
         # Hermitian Gram of centered basin members. For complex (FHRR)
         # tensors, diffs.conj().T @ diffs is Hermitian → real eigenvalues
         # via torch.linalg.eigh.
-        sigma = (diffs.conj().transpose(-1, -2) @ diffs) / float(n)
+        # Compute the eigenvalues of σ = diffs.conj().T @ diffs / n via the
+        # n×n Gram matrix gram = diffs @ diffs.conj().T / n instead of the
+        # D×D scatter matrix. The two matrices share exactly the same set
+        # of non-zero eigenvalues (standard "kernel trick" identity); the
+        # D×D form additionally carries (D - n) trivial zero eigenvalues
+        # because rank(σ) ≤ n_members ≤ basin_trace_buffer_size (64) ≪ D
+        # (4096 by default in this project). That (D - n) zero subspace
+        # makes σ numerically ill-conditioned at the precision available
+        # to torch.linalg.eigvalsh — observed on Colab CUDA at 2026-05-27
+        # as LinAlgError 4095 and even on CPU LAPACK as LinAlgError 5/12.
+        # The n×n Gram path is full-rank for non-degenerate samples and
+        # an order of magnitude smaller (4 KB vs 16 MB at D=4096, n=8).
+        # Mathematically byte-identical at the λ_1 / λ_2 layer used below;
+        # the C.2.2 dynamic's behavior is unchanged.
+        gram = (diffs @ diffs.conj().transpose(-1, -2)) / float(n)
         # Eigh returns ascending eigenvalues. Take top two: λ_1 (last),
         # λ_2 (second-to-last). All ops stay on-device.
-        eigvals = torch.linalg.eigvalsh(sigma)
+        try:
+            eigvals = torch.linalg.eigvalsh(gram)
+        except torch._C._LinAlgError:
+            # Defensive: keep the CPU fallback in case some pathological
+            # input still trips cuSOLVER (e.g. identical basin members).
+            eigvals = torch.linalg.eigvalsh(gram.cpu()).to(gram.device)
         lam_1 = eigvals[-1]
         lam_2 = eigvals[-2] if eigvals.shape[0] >= 2 else torch.zeros_like(lam_1)
         # Clamp at 0 — eigh may return tiny negatives for near-singular Σ.
diff --git a/src/energy_memory/phase2/corpus.py b/src/energy_memory/phase2/corpus.py
--- a/src/energy_memory/phase2/corpus.py
+++ b/src/energy_memory/phase2/corpus.py
@@ -113,7 +113,13 @@ def load_repo_sample_splits(repo_root: Path) -> Dict[str, List[str]]:
 def load_wikitext_splits(name: str = "wikitext-2-raw-v1") -> Dict[str, List[str]]:
     if load_dataset is None:  # pragma: no cover - exercised only when dependency missing
         raise ModuleNotFoundError("datasets is required to load WikiText-2")
-    dataset = load_dataset("wikitext", name)
+    # Use the canonical Salesforce/wikitext namespace. The bare "wikitext"
+    # form worked with older HF stacks but recent huggingface_hub versions
+    # (~0.30+) ship a stricter HF URI parser that rejects any repo id
+    # without an explicit namespace, raising HfUriError. The Salesforce
+    # mirror is the current canonical home of the dataset; config names
+    # ("wikitext-2-raw-v1", "wikitext-103-raw-v1", ...) are unchanged.
+    dataset = load_dataset("Salesforce/wikitext", name)
     return {
         "train": [row["text"] for row in dataset["train"]],
         "validation": [row["text"] for row in dataset["validation"]],
'''

with open('/tmp/c3_combined.patch', 'w') as f:
    f.write(patch)
check = subprocess.run(['git', 'apply', '--check', '/tmp/c3_combined.patch'], capture_output=True, text=True)
if check.returncode == 0:
    subprocess.check_call(['git', 'apply', '/tmp/c3_combined.patch'])
    print('combined patch applied.')
else:
    if int(pre_pull or '0') >= 1 and int(pre_gram or '0') >= 1 and int(pre_wt or '0') >= 1:
        print('all three patches already in branch — skipping apply.')
    else:
        print('PATCH APPLY FAILED:'); print(check.stderr)
        raise SystemExit('Cannot continue.')

post_pull = subprocess.check_output(['grep', '-c', '--', '--lr-pull', 'experiments/c3_phase3_exit_criterion.py']).decode().strip()
post_gram = subprocess.check_output(['grep', '-c', 'kernel trick', 'src/energy_memory/phase4/consolidation.py']).decode().strip()
post_wt   = subprocess.check_output(['grep', '-c', 'Salesforce/wikitext', 'src/energy_memory/phase2/corpus.py']).decode().strip()
print(f'post-patch tracers: --lr-pull = {post_pull}; kernel-trick = {post_gram}; Salesforce/wikitext = {post_wt}')
assert int(post_pull) >= 1 and int(post_gram) >= 1 and int(post_wt) >= 1, 'patches missing'

In [ ]:
# 2. Mount Drive.
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/neuro-ai/results', exist_ok=True)
print('Drive mounted.')

In [ ]:
# 3. Install deps. `datasets<3` pinned + Salesforce/wikitext patch in (a) handles HF compat.
!pip install -q "datasets<3"
import sys, torch, numpy as np, datasets
print(f'python: {sys.version.split()[0]} | torch: {torch.__version__} | numpy: {np.__version__} | datasets: {datasets.__version__}')
print(f'cuda available: {torch.cuda.is_available()}; device 0: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')

In [ ]:
# 4. Pre-warm wikitext-2 cache so 30 subprocesses share the on-disk HF cache.
import sys; sys.path.insert(0, '/content/Neuro-AI/src')
from energy_memory.phase2.corpus import load_corpus_splits
from pathlib import Path
print('warming wikitext-2-raw-v1 cache (parent process, CPU only)...')
splits = load_corpus_splits('wikitext', Path('/content/Neuro-AI'), wikitext_name='wikitext-2-raw-v1')
print(f'  train: {len(splits["train"])} rows')
print(f'  validation: {len(splits["validation"])} rows')
print(f'  test: {len(splits["test"])} rows')
print('cache warmed.')
del splits; import gc; gc.collect()

In [ ]:
# 5. GPU info.
!nvidia-smi --query-gpu=name,memory.total,compute_mode --format=csv
!nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv

In [ ]:
# 6. SMOKE — one tiny subprocess to confirm the runtime + patches are sane.
import subprocess, sys, os
from pathlib import Path
os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'
smoke_out = Path('reports/c3_robustness_v4_smoke_2026-05-27')
smoke_out.mkdir(parents=True, exist_ok=True)
smoke_log = Path('reports/c3_robustness_v4_smoke.log')
cmd = [sys.executable, 'experiments/c3_phase3_exit_criterion.py',
       '--seeds', '0', '--device', 'cuda',
       '--lr-pull', '0.1', '--lr-push', '0.05',
       '--n-consolidation-events', '100',
       '--alpha-anti', '0.01', '--repulsion-step-size', '0.05',
       '--output-dir', str(smoke_out)]
with smoke_log.open('w') as logf:
    rc = subprocess.call(cmd, stdout=logf, stderr=subprocess.STDOUT)
print(f'smoke exit code: {rc}; json: {(smoke_out / "c3_summary.json").exists()}')
print('\n=== smoke log (last 40 lines) ===')
!tail -40 {smoke_log}
if rc != 0:
    raise SystemExit('Smoke failed — abort.')
print('\nSmoke OK.')

In [ ]:
# 7. PARALLEL launch — 3 robustness probes × 10 seeds = 30 subprocesses.
import subprocess, os, time, signal, sys
from pathlib import Path
os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'
PY = sys.executable

# (tag, lr_pull, n_events, corpus_source, beta, D, seeds_list)
PROBES = [
    ('replicate_seeds10_19', 0.1, 1000, 'wikitext', 10.0, 4096, list(range(10, 20))),
    ('beta30_seeds0_9',      0.1, 1000, 'wikitext', 30.0, 4096, list(range(10))),
    ('D2048_seeds0_9',       0.1, 1000, 'wikitext', 10.0, 2048, list(range(10))),
]

# Flatten into per-seed subprocesses (30 entries).
ENTRIES = []
for tag, lp, ne, corp, beta, D, seeds in PROBES:
    for seed in seeds:
        ENTRIES.append((tag, lp, ne, corp, beta, D, seed))
print(f'launching {len(ENTRIES)} per-seed subprocesses ({len(PROBES)} probes × 10 seeds each)')

log_root = Path('reports/c3_robustness_v4_logs')
log_root.mkdir(parents=True, exist_ok=True)

def out_dir_for(tag, seed):
    return f'reports/c3_robustness_v4_{tag}_seed{seed}_2026-05-27'

def launch(tag, lr_pull, n_events, corpus_source, beta, D, seed):
    out_dir = out_dir_for(tag, seed)
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    log_path = log_root / f'{tag}_seed{seed}.log'
    logf = open(log_path, 'w')
    cmd = [PY, 'experiments/c3_phase3_exit_criterion.py',
           '--seeds', str(seed), '--device', 'cuda',
           '--lr-pull', str(lr_pull), '--lr-push', '0.05',
           '--n-consolidation-events', str(n_events),
           '--alpha-anti', '0.01', '--repulsion-step-size', '0.05',
           '--corpus-source', corpus_source,
           '--beta', str(beta),
           '--D', str(D),
           '--output-dir', out_dir]
    proc = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT)
    return proc, logf, out_dir, log_path

def snapshot(remaining, total, t0):
    elapsed = (time.time() - t0) / 60
    n_done = total - len(remaining)
    print(f'  --- snapshot at {elapsed:.1f} min — {n_done}/{total} done, {len(remaining)} running ---')
    try:
        gpu = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=memory.used,utilization.gpu', '--format=csv,noheader'],
            stderr=subprocess.DEVNULL).decode().strip()
        print(f'  GPU: {gpu}')
    except Exception as e:
        print(f'  GPU snapshot failed: {e}')
    from collections import Counter
    cond_running = Counter()
    for key in remaining:
        cond_running[key.rsplit('_seed', 1)[0]] += 1
    for spec in PROBES:
        tag = spec[0]
        n_run = cond_running.get(tag, 0)
        n_done_tag = 10 - n_run
        print(f'    {tag:>22}: {n_done_tag}/10 done')

def kill_all(remaining):
    for key, (proc, logf, _, _) in remaining.items():
        try:
            proc.send_signal(signal.SIGKILL); logf.close()
        except Exception:
            pass

# Staggered launch (1.5 s × 30 = 45 s).
procs = {}
for entry in ENTRIES:
    tag, _, _, _, _, _, seed = entry
    key = f'{tag}_seed{seed}'
    procs[key] = launch(*entry)
    time.sleep(1.5)
print(f'all {len(procs)} cells launched  ({time.strftime("%H:%M:%S")})')

t0 = time.time()
remaining = dict(procs)
total = len(procs)
failures = []
poll_count = 0
try:
    while remaining:
        done_this_round = []
        for key, (proc, logf, out_dir, log_path) in remaining.items():
            rc = proc.poll()
            if rc is not None:
                logf.close()
                elapsed = (time.time() - t0) / 60
                json_exists = Path(out_dir, 'c3_summary.json').exists()
                ok = 'OK' if rc == 0 else f'FAILED (exit={rc})'
                print(f'  [{elapsed:5.1f} min] {key:>32}: {ok}  json={json_exists}')
                if rc != 0:
                    failures.append(key)
                    print(f'    --- last 30 lines of {log_path} ---')
                    try:
                        out = subprocess.check_output(['tail', '-30', str(log_path)],
                            stderr=subprocess.DEVNULL).decode()
                        for line in out.splitlines():
                            print(f'    | {line}')
                    except Exception as e:
                        print(f'    | (could not read log: {e})')
                    print('    --- end log ---')
                done_this_round.append(key)
        for key in done_this_round:
            del remaining[key]
        if remaining:
            poll_count += 1
            if poll_count % 3 == 0:
                snapshot(remaining, total, t0)
            time.sleep(30)
except KeyboardInterrupt:
    print('\n!!! Interrupted !!!')
    kill_all(remaining); raise

print(f'\nALL DONE in {(time.time()-t0)/60:.1f} min')
print(f'failures: {len(failures)}/{total}')
if failures:
    print('  failed keys:', failures)
!nvidia-smi --query-gpu=memory.used,memory.total,utilization.gpu --format=csv

In [ ]:
# 7b. EMERGENCY kill.
import subprocess, signal, os
killed = 0
for line in subprocess.check_output(['ps', '-eo', 'pid,cmd']).decode().splitlines():
    if 'c3_phase3_exit_criterion' in line and 'grep' not in line:
        try:
            pid = int(line.split()[0])
            os.kill(pid, signal.SIGKILL); print(f'  killed {pid}'); killed += 1
        except Exception as e:
            print(f'  err: {e}')
print(f'killed {killed} workers')

In [ ]:
# 8. Copy results + logs to Drive.
import shutil, os
dst_root = '/content/drive/MyDrive/neuro-ai/results/c3_robustness_v4_2026-05-27'
os.makedirs(dst_root, exist_ok=True)
TAGS = ['replicate_seeds10_19', 'beta30_seeds0_9', 'D2048_seeds0_9']
count = 0
for tag in TAGS:
    # The seeds for replicate_ are 10..19; for others 0..9
    seeds = list(range(10, 20)) if tag == 'replicate_seeds10_19' else list(range(10))
    for seed in seeds:
        src = f'reports/c3_robustness_v4_{tag}_seed{seed}_2026-05-27'
        if os.path.isdir(src):
            shutil.copytree(src, f'{dst_root}/{tag}_seed{seed}', dirs_exist_ok=True)
            count += 1
if os.path.isdir('reports/c3_robustness_v4_logs'):
    shutil.copytree('reports/c3_robustness_v4_logs', f'{dst_root}/colab_logs', dirs_exist_ok=True)
print(f'copied {count} per-seed dirs + logs to {dst_root}')
!ls {dst_root} | head -20

In [ ]:
# 9. AGGREGATION — merge per-seed JSONs per probe and apply the robustness criterion.
import json, math
from pathlib import Path

PROBES = [
    # (tag, beta, D, seeds_list, expected_n_seeds)
    ('replicate_seeds10_19', 10.0, 4096, list(range(10, 20)), 10),
    ('beta30_seeds0_9',      30.0, 4096, list(range(10)),      10),
    ('D2048_seeds0_9',       10.0, 2048, list(range(10)),      10),
]
STRATA = ('tight', 'spread', 'borderline')

def wilson(successes, trials, z=1.96):
    if trials == 0: return (0.0, 0.0, 0.0)
    p = successes / trials; n = trials
    denom = 1 + z*z/n
    center = (p + z*z/(2*n)) / denom
    half = (z * math.sqrt(p*(1-p)/n + z*z/(4*n*n))) / denom
    return (p, max(0.0, center - half), min(1.0, center + half))

def merge_tag(tag, seeds):
    rows = []
    seeds_present = set()
    modes_seen = set()
    for seed in seeds:
        p = Path(f'reports/c3_robustness_v4_{tag}_seed{seed}_2026-05-27/c3_summary.json')
        if not p.exists():
            continue
        d = json.loads(p.read_text())
        seeds_present.add(seed)
        for r in d['per_cell_rows']:
            rows.append(r)
            modes_seen.add(r['theta_prime_mode'])
    aggregated = {}
    for mode in sorted(modes_seen):
        aggregated[mode] = {}
        for is_control in (False, True):
            key = 'shuffled_control' if is_control else 'standard'
            aggregated[mode][key] = {}
            for stratum in STRATA:
                tot_s = 0; tot_t = 0
                for r in rows:
                    if r['theta_prime_mode'] != mode or r['is_control'] != is_control: continue
                    cell = r['per_stratum'][stratum]
                    tot_s += int(cell['successes']); tot_t += int(cell['trials'])
                mean_v, lo, hi = wilson(tot_s, tot_t)
                aggregated[mode][key][stratum] = {'successes': tot_s, 'trials': tot_t,
                    'recall_at_k': mean_v, 'wilson_lower': lo, 'wilson_upper': hi}
        aggregated[mode]['delta_standard_minus_control'] = {}
        for stratum in STRATA:
            s = aggregated[mode]['standard'][stratum]; c = aggregated[mode]['shuffled_control'][stratum]
            aggregated[mode]['delta_standard_minus_control'][stratum] = {
                'delta_recall_at_k': s['recall_at_k'] - c['recall_at_k'],
                'standard_trials': s['trials'], 'control_trials': c['trials'],
                'ci_disjoint_standard_beats_control': s['wilson_lower'] > c['wilson_upper']}
    return aggregated, sorted(seeds_present), sorted(modes_seen)

import os
merged_root = '/content/drive/MyDrive/neuro-ai/results/c3_robustness_v4_2026-05-27/_merged_n10'
os.makedirs(merged_root, exist_ok=True)

print(f'{"probe":>22} {"β":>5} {"D":>5} {"seeds":>5} {"mode":>11} {"stratum":>11}  '
      f'{"std R@K":>22}  {"ctrl R@K":>22}  {"Δ":>7}  disjoint?  n_std  n_ctrl')
probe_disjoint = {}  # tag → has_disjoint
for tag, beta, D, seeds, exp_seeds in PROBES:
    agg, seeds_present, modes = merge_tag(tag, seeds)
    if not seeds_present:
        print(f'{tag:>22} {beta:>5} {D:>5}  ALL MISSING')
        probe_disjoint[tag] = False; continue
    if len(seeds_present) < exp_seeds:
        print(f'  (warning: {tag} has only {len(seeds_present)}/{exp_seeds} seeds present: {seeds_present})')
    out = {'tag': tag, 'beta': beta, 'D': D, 'seeds_present': seeds_present, 'aggregated': agg}
    with open(f'{merged_root}/{tag}.json', 'w') as f:
        json.dump(out, f, indent=2)
    has_disjoint = False
    for mode in modes:
        for stratum in STRATA:
            s = agg[mode]['standard'][stratum]; c = agg[mode]['shuffled_control'][stratum]
            dl = agg[mode]['delta_standard_minus_control'][stratum]
            if s['trials'] == 0 and c['trials'] == 0: continue
            std_str  = f'{s["recall_at_k"]:.3f} [{s["wilson_lower"]:.3f},{s["wilson_upper"]:.3f}]'
            ctrl_str = f'{c["recall_at_k"]:.3f} [{c["wilson_lower"]:.3f},{c["wilson_upper"]:.3f}]'
            disj = 'YES' if dl['ci_disjoint_standard_beats_control'] else 'no'
            print(f'{tag:>22} {beta:>5} {D:>5} {len(seeds_present):>5} {mode:>11} {stratum:>11}  '
                  f'{std_str:>22}  {ctrl_str:>22}  {dl["delta_recall_at_k"]:>+7.3f}  {disj:>9}  '
                  f'{s["trials"]:>5}  {c["trials"]:>5}')
            if dl['ci_disjoint_standard_beats_control'] and s['trials'] > 0 and c['trials'] > 0:
                has_disjoint = True
    probe_disjoint[tag] = has_disjoint

n_pass = sum(probe_disjoint.values())
print()
print(f'=== ROBUSTNESS SUMMARY: {n_pass}/3 probes reproduce CI-disjoint signal ===')
for tag, passed in probe_disjoint.items():
    mark = '✓' if passed else '✗'
    print(f'  {mark} {tag}')
print()
if n_pass == 3:
    print('VERDICT: Phase 3 graduation LOCKED. Robust across seed set, β, and D.')
    print('  → Write Report 112, update STATUS.md to reopen Phase 5′.')
elif n_pass == 2:
    print('VERDICT: Phase 3 graduation HOLDS with one operating-point caveat.')
    print('  → Write Report 112; narrow the operating-point claim to the dimensions that passed.')
elif n_pass == 1:
    print('VERDICT: Operating-point-specific finding, not graduation-robust.')
    print('  → Reframe Report 112 as an operating-point sensitivity study.')
else:
    print('VERDICT: v3 result was artifact. DO NOT graduate.')
    print('  → Investigate before any Phase 5′ pivot.')